# Whisker-Pole Contact Classifier — Training

Fine-tune an **EfficientNet-B3** (~14 M params) for binary classification:
- **1** = whisker contacts pole
- **0** = no contact

**Features:**
- ImageNet-pretrained backbone
- Albumentations augmentation (flip, rotate, brightness, blur, cutout)
- Binary Cross-Entropy with Logits loss
- Cosine-annealing LR schedule
- Early stopping on validation loss
- Saves best model checkpoint

In [ ]:
import sys, os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

TRAINING_DIR = os.path.dirname(os.path.abspath("__file__"))
if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)

from dataset import ContactDataset, get_train_transforms, get_val_transforms
from trainer import (
    build_model,
    count_parameters,
    train_one_epoch,
    evaluate,
    EarlyStopping,
    save_checkpoint,
)

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1 — Configuration

In [ ]:
# ========================  PATHS  ========================

# Pickle files (on Windows side, accessed via WSL)
DATA_DIR = "/mnt/c/Users/wanglab/Desktop/Club Like Endings/102725_1/data"
TRAIN_PKL = os.path.join(DATA_DIR, "train.pkl")
TEST_PKL  = os.path.join(DATA_DIR, "test.pkl")

# Where to save model checkpoints
SAVE_DIR = os.path.join(TRAINING_DIR, "checkpoints")
os.makedirs(SAVE_DIR, exist_ok=True)

# ========================  HYPERPARAMETERS  ========================

IMG_SIZE       = 256       # resize from 640x480 → 256x256
BATCH_SIZE     = 32
NUM_WORKERS    = 4
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-4
EPOCHS         = 50
PATIENCE       = 10        # early stopping patience
FREEZE_BACKBONE = False    # set True for staged fine-tuning

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

## 2 — Load Datasets & DataLoaders

In [ ]:
train_ds = ContactDataset(TRAIN_PKL, transform=get_train_transforms(IMG_SIZE))
val_ds   = ContactDataset(TEST_PKL,  transform=get_val_transforms(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples : {len(train_ds):,}")
print(f"Val samples   : {len(val_ds):,}")
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

In [ ]:
# Quick visual sanity check — show a batch of augmented images
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(14, 6))
mean = np.array([0.485, 0.456, 0.406])
std  = np.array([0.229, 0.224, 0.225])

for i, ax in enumerate(axes.flatten()):
    if i >= len(images):
        break
    img = images[i].permute(1, 2, 0).numpy()  # C,H,W → H,W,C
    img = np.clip(img * std + mean, 0, 1)      # un-normalize
    lbl = int(labels[i].item())
    ax.imshow(img)
    ax.set_title("Contact" if lbl else "No Contact",
                 color="red" if lbl else "blue")
    ax.axis("off")

plt.suptitle("Augmented Training Batch", fontweight="bold")
plt.tight_layout()
plt.show()

## 3 — Build Model

In [ ]:
model = build_model(
    num_classes=1,
    pretrained=True,
    dropout=0.3,
    freeze_backbone=FREEZE_BACKBONE,
).to(DEVICE)

params = count_parameters(model)
print(f"Total params     : {params['total']:,}")
print(f"Trainable params : {params['trainable']:,}")

## 4 — Loss, Optimizer, Scheduler

In [ ]:
# Compute class weight for imbalanced data
n_pos = (train_ds.labels == 1).sum()
n_neg = (train_ds.labels == 0).sum()
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float32).to(DEVICE)
print(f"Class counts — Contact: {n_pos}, No Contact: {n_neg}")
print(f"pos_weight: {pos_weight.item():.2f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

early_stopping = EarlyStopping(patience=PATIENCE, mode="min")

## 5 — Training Loop

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
ckpt_path = os.path.join(SAVE_DIR, "contact_classifier.pt")

config = {
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "freeze_backbone": FREEZE_BACKBONE,
    "model_arch": "efficientnet_b3",
}

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}  (lr={optimizer.param_groups[0]['lr']:.2e})")

    train_metrics = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, scheduler)
    val_metrics   = evaluate(model, val_loader, criterion, DEVICE)

    history["train_loss"].append(train_metrics["loss"])
    history["val_loss"].append(val_metrics["loss"])
    history["train_acc"].append(train_metrics["accuracy"])
    history["val_acc"].append(val_metrics["accuracy"])

    print(f"  Train — loss: {train_metrics['loss']:.4f}  acc: {train_metrics['accuracy']:.4f}")
    print(f"  Val   — loss: {val_metrics['loss']:.4f}  acc: {val_metrics['accuracy']:.4f}  "
          f"f1: {val_metrics['f1']:.4f}  prec: {val_metrics['precision']:.4f}  "
          f"rec: {val_metrics['recall']:.4f}")

    # Early stopping check — saves checkpoint to disk when loss improves
    improved = early_stopping(val_metrics["loss"], model)
    if improved:
        save_checkpoint(model, optimizer, epoch, val_metrics["loss"],
                        history, config, ckpt_path)

    if early_stopping.should_stop:
        print(f"\n⏹  Early stopping triggered after {epoch} epochs.")
        break

# Restore best model weights
if early_stopping.best_model_state is not None:
    model.load_state_dict(early_stopping.best_model_state)
    print(f"\nRestored best model (val_loss = {early_stopping.best_score:.4f})")
    print(f"Best checkpoint saved at: {ckpt_path}")